In [ ]:
 pip install unbabel-comet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 90.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 64.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.2 MB/s

Calculate COMET score

In [ ]:
from comet import download_model, load_from_checkpoint

SRC_FILE = "/path/taj_ne.src"      # Path to source sentences
HYP_FILE = "/path/taj_ne.hyp"  # Path to hypothesis (model translations)
REF_FILE = "/path/taj_ne.ref"   # Path to reference translations
MODEL_NAME = "Unbabel/wmt22-comet-da"
SAVE_SEGMENT_SCORES = True
OUTPUT_FILE = "comet_segment_scores.txt"

def read_file(filepath):

    with open(filepath, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f]

def calculate_comet_score(src_file, hyp_file, ref_file, model_name):

    # Read files
    print("Reading input files...")
    sources = read_file(src_file)
    hypotheses = read_file(hyp_file)
    references = read_file(ref_file)

    # Check file lengths match
    if not (len(sources) == len(hypotheses) == len(references)):
        raise ValueError(
            f"   File lengths don't match!\n"
            f"   Source: {len(sources)} lines\n"
            f"   Hypothesis: {len(hypotheses)} lines\n"
            f"   Reference: {len(references)} lines"
        )

    print(f"  All files loaded successfully ({len(sources)} sentences)")

    # Download and load model
    print(f"\n  Loading COMET model: {model_name}")
    model_path = download_model(model_name)
    model = load_from_checkpoint(model_path)
    print(" Model loaded successfully")

    # Prepare data for COMET
    data = []
    for src, hyp, ref in zip(sources, hypotheses, references):
        data.append({
            "src": src,
            "mt": hyp,
            "ref": ref
        })

    # Calculate scores
    print("\n  Calculating COMET scores...")
    output = model.predict(data, batch_size=16, gpus=1)

    # Extract results
    scores = output.scores
    system_score = output.system_score

    print(" Calculation complete!")

    return {
        "system_score": system_score,
        "segment_scores": scores,
        "num_segments": len(scores)
    }

def main():
    """Main function to calculate and display COMET scores."""
    try:
        # Calculate COMET scores
        results = calculate_comet_score(SRC_FILE, HYP_FILE, REF_FILE, MODEL_NAME)

        # Print results
        print("\n" + "="*60)
        print("  COMET SCORE RESULTS")
        print("="*60)
        print(f" System Score:        {results['system_score']:.4f}")
        print(f" Number of segments:  {results['num_segments']}")
        print(f" Min segment score:   {min(results['segment_scores']):.4f}")
        print(f" Max segment score:   {max(results['segment_scores']):.4f}")
        print(f" Avg segment score:   {sum(results['segment_scores'])/len(results['segment_scores']):.4f}")
        print("="*60)

        # Save segment scores if requested
        if SAVE_SEGMENT_SCORES:
            with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
                for score in results['segment_scores']:
                    f.write(f"{score:.4f}\n")
            print(f"\n Segment scores saved to: {OUTPUT_FILE}")

        # Display score interpretation
        print("\n  Score Interpretation:")
        print("   > 0.80: Excellent translation quality")
        print("   0.60-0.80: Good translation quality")
        print("   0.40-0.60: Moderate translation quality")
        print("   < 0.40: Poor translation quality")

        return results

    except FileNotFoundError as e:
        print(f"\n Error: Could not find file - {e}")
        print("Please check your file paths in the configuration section.")
    except Exception as e:
        print(f"\n Error occurred: {e}")
        raise

# Run the calculation
if __name__ == "__main__":
    results = main()

📁 Reading input files...
✅ All files loaded successfully (5000 sentences)

🔄 Loading COMET model: Unbabel/wmt22-comet-da


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/pytorch_lightning/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


✅ Model loaded successfully

🧮 Calculating COMET scores...


Predicting DataLoader 0: 100%|██████████| 313/313 [01:48<00:00,  2.87it/s]


✅ Calculation complete!

🎯 COMET SCORE RESULTS
📊 System Score:        0.7737
📝 Number of segments:  5000
📉 Min segment score:   0.3221
📈 Max segment score:   0.9909
📊 Avg segment score:   0.7737

💾 Segment scores saved to: comet_segment_scores.txt

📖 Score Interpretation:
   > 0.80: Excellent translation quality
   0.60-0.80: Good translation quality
   0.40-0.60: Moderate translation quality
   < 0.40: Poor translation quality


In [ ]:
!pip install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 4.2 MB/s eta 0:00:00


Calculate CHRF score

In [ ]:
import sacrebleu

def calculate_chrf(hyp_file, ref_file):
    """
    Calculate chrF++ score between hypothesis and reference translations.

    Args:
        hyp_file: Path to hypothesis file (.txt)
        ref_file: Path to reference file (.txt)

    Returns:
        chrF++ score
    """
    # Read hypothesis file
    with open(hyp_file, 'r', encoding='utf-8') as f:
        hypotheses = [line.strip() for line in f]

    # Read reference file
    with open(ref_file, 'r', encoding='utf-8') as f:
        references = [line.strip() for line in f]

    # Check if files have same number of lines
    if len(hypotheses) != len(references):
        raise ValueError(f"Number of lines don't match: {len(hypotheses)} hyp vs {len(references)} ref")

    # Calculate chrF++ score
    # chrF++ uses word n-grams in addition to character n-grams
    chrf = sacrebleu.corpus_chrf(hypotheses, [references], word_order=2)

    print(f"Number of sentences: {len(hypotheses)}")
    print(f"chrF++ score: {chrf.score:.2f}")
    print(f"\nDetailed score: {chrf}")

    return chrf.score

if __name__ == "__main__":
    # Example usage
    hyp_file = "/path/taj_ne.hyp"  # Replace with your hypothesis file path
    ref_file = "/path/taj_ne.ref"    # Replace with your reference file path

    try:
        score = calculate_chrf(hyp_file, ref_file)
    except FileNotFoundError as e:
        print(f"Error: File not found - {e}")
    except Exception as e:
        print(f"Error: {e}")

Number of sentences: 5000
chrF++ score: 66.30

Detailed score: chrF2++ = 66.30
